# 02 — Preprocesamiento y Feature Engineering
**CRISP-DM: Preparación de los datos** · Etapa 1: *Preprocesamiento y feature engineering*

Limpieza, tratamiento de outliers, resampleo semanal, relleno de huecos,
integración de las 4 fuentes externas y creación de características.

**Tratamiento de datos nulos (resumen):** se aplica un método distinto según el
origen del nulo — interpolación lineal para el precio (serie continua),
`ffill`/`bfill` para las fuentes externas (variables de variación lenta o de
menor frecuencia), y descarte explícito de filas para el NaN estructural que
generan los lags/medias móviles al inicio de cada serie (no se imputa, para no
inventar historia inexistente). Cada etapa se diagnostica con
`pp.diagnosticar_nulos` para dejar evidencia de qué se corrigió.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))  # importar el paquete sin instalar
import pandas as pd
from prediccion_precios import data_loading as dl, preprocessing as pp, features as ft, eda, config
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)

## 1. Precios objetivo → limpieza → outliers

In [2]:
precios = dl.filtrar_productos_objetivo(dl.cargar_precios_consolidados())
precios = pp.limpiar_precios(precios)
precios = pp.tratar_outliers(precios)      # winsorización IQR por producto
precios.shape

(3610, 4)

In [3]:
pp.diagnosticar_nulos(precios, "precios diarios (crudo, tras limpieza/outliers)")

--- Nulos: precios diarios (crudo, tras limpieza/outliers) (3610 filas) ---
        n_nulos  pct_nulos
Precio       11        0.3


,n_nulos,pct_nulos
Precio,11,0.3


## 2. Resampleo semanal + rejilla continua (interpolación)

**Tratamiento de nulos — Precio:** el consolidado diario tiene ~0.3% de precios
faltantes por producto objetivo (ver `config.PRODUCTOS_OBJETIVO`), y al
resamplear a semanas aparecen además semanas sin ningún registro diario
(huecos de mercado). Se reindexa cada producto a una rejilla semanal continua
y se interpola linealmente (`tratar_faltantes`), incluyendo los extremos
(`limit_direction="both"`). Es el método adecuado porque el precio es una
serie temporal con tendencia suave semana a semana: interpolar respeta esa
tendencia sin inventar saltos bruscos ni descartar semanas (crítico para no
romper la rejilla que usan los lags/medias móviles más adelante).

In [4]:
sem = pp.resamplear_frecuencia(precios)     # media semanal por producto
pp.diagnosticar_nulos(sem, "precios semanales (antes de interpolar)")
sem = pp.tratar_faltantes(sem)             # reindexa a rejilla semanal completa + interpola linealmente
pp.diagnosticar_nulos(sem, "precios semanales (después de interpolar)")

--- Nulos: precios semanales (antes de interpolar) (825 filas) ---
        n_nulos  pct_nulos
Precio       40       4.85
--- Nulos: precios semanales (después de interpolar) (825 filas) ---
Sin nulos.


,n_nulos,pct_nulos


## 3. Integración de fuentes externas
combustible (diario), precipitación (diario), IPC transporte (mensual), producción FAOSTAT (anual).

**Tratamiento de nulos — fuentes externas:** al unir por fecha/mes-año/año-cultivo
quedan NaN en los extremos de cada serie (semanas anteriores al primer dato o
posteriores al último de cada fuente). `integrar_fuentes` los rellena con
`ffill().bfill()` por producto (propaga el valor vigente más cercano),
apropiado porque son variables de entorno de variación lenta (combustible,
IPC, precipitación acumulada) o de frecuencia menor a la semanal (IPC
mensual, producción anual): no tiene sentido interpolar un valor mensual/anual
dentro de una semana, mejor mantener el último dato observado.

In [5]:
comb = dl.cargar_combustible()
prec = dl.cargar_precipitacion()
ipc  = dl.cargar_ipc()
prod = dl.cargar_produccion()
data = pp.integrar_fuentes(sem, combustible=comb, precipitacion=prec, ipc=ipc, produccion=prod)
print(data.shape); data.head(3)

(825, 15)


,Fecha,Precio,Producto,Grupo,Anio,Mes,diesel,gas_regular,gas_especial,precipitacion_mm,lluvia_max_mm,ipc_transporte,area_ha,produccion_t,rendimiento_kg_ha
0,2021-05-09,38.615385,ARROZ ORO PRIMERA CLASE IMPORTADO,ARROZ,2021,5,2.932381,3.461905,3.650000,37.323,69.4,111.72,2836.0,19145.98,6751.2
1,2021-05-16,38.898088,ARROZ ORO PRIMERA CLASE IMPORTADO,ARROZ,2021,5,2.940000,3.466667,3.656667,16.133,85.2,111.72,2836.0,19145.98,6751.2
2,2021-05-23,38.927532,ARROZ ORO PRIMERA CLASE IMPORTADO,ARROZ,2021,5,3.031429,3.518095,3.710952,20.783,73.0,111.72,2836.0,19145.98,6751.2


In [6]:
pp.diagnosticar_nulos(data, "tras integrar fuentes externas (ffill/bfill)")

--- Nulos: tras integrar fuentes externas (ffill/bfill) (825 filas) ---
Sin nulos.


,n_nulos,pct_nulos


## 4. Feature engineering
calendario cíclico, lags, medias/desv móviles y temporada de cosecha (aporte local).

**Nulos estructurales:** los lags (`lag1..lag8`) y las medias/desviaciones
móviles requieren semanas previas; en las primeras 1-8 semanas de cada
producto no existen aún y quedan en NaN. No se imputan aquí (imputar
inventaría historia de precio inexistente) — se documentan y se descartan
explícitamente antes de modelar, en el notebook 03
(`features.construir_matriz_modelado(data, dropna=True)`), para que el
descarte quede visible y cuantificado en vez de ser implícito.

In [7]:
data = ft.agregar_variables_calendario(data)
data = ft.agregar_lags(data)
data = ft.agregar_medias_moviles(data)
data = ft.marcar_temporada_cosecha(data)
print(data.shape); list(data.columns)

(825, 33)


['Fecha',
 'Precio',
 'Producto',
 'Grupo',
 'Anio',
 'Mes',
 'diesel',
 'gas_regular',
 'gas_especial',
 'precipitacion_mm',
 'lluvia_max_mm',
 'ipc_transporte',
 'area_ha',
 'produccion_t',
 'rendimiento_kg_ha',
 'mes',
 'trimestre',
 'semana_anio',
 'mes_sin',
 'mes_cos',
 'semana_sin',
 'semana_cos',
 'precio_lag1',
 'precio_lag2',
 'precio_lag4',
 'precio_lag8',
 'media_movil_4',
 'desv_movil_4',
 'media_movil_8',
 'desv_movil_8',
 'media_movil_12',
 'desv_movil_12',
 'es_cosecha']

In [8]:
pp.diagnosticar_nulos(data, "tras features (lags/medias móviles) — NaN estructural esperado")

--- Nulos: tras features (lags/medias móviles) — NaN estructural esperado (825 filas) ---
                n_nulos  pct_nulos
precio_lag8          40       4.85
precio_lag4          20       2.42
desv_movil_12        10       1.21
precio_lag2          10       1.21
desv_movil_4         10       1.21
desv_movil_8         10       1.21
precio_lag1           5       0.61
media_movil_4         5       0.61
media_movil_8         5       0.61
media_movil_12        5       0.61


,n_nulos,pct_nulos
precio_lag8,40,4.85
precio_lag4,20,2.42
desv_movil_12,10,1.21
precio_lag2,10,1.21
desv_movil_4,10,1.21
desv_movil_8,10,1.21
precio_lag1,5,0.61
media_movil_4,5,0.61
media_movil_8,5,0.61
media_movil_12,5,0.61


## 5. Guardar dataset de modelado

Se guarda **con** los NaN estructurales de los lags/medias móviles iniciales
(no se eliminan aquí), para no perder columnas/filas útiles fuera del
contexto directo de modelado. El descarte de esas filas ocurre explícitamente
en el notebook 03, justo antes de entrenar (`construir_matriz_modelado`).

In [9]:
ruta = pp.guardar_processed(data)
print('Guardado en', ruta)

Guardado en C:\Users\irmal\Downloads\prediccion_precios_agricolas_V2\prediccion\data\processed\dataset_modelado.csv
